In [0]:
%run ./helper

In [0]:
#dbutils.widgets.removeAll()

In [0]:
source_table_list=["topic_info_local"]
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)


#dbutils.widgets.dropdown("model_result_table", model_result_table_list[0], model_result_table_list)
dbutils.widgets.dropdown("instruct_model", instruct_model_list[0], instruct_model_list)
dbutils.widgets.dropdown("instruction_activity", instruction_activity_list[0], instruction_activity_list)
catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
#model_result_table=dbutils.widgets.get("model_result_table")
instruct_model=dbutils.widgets.get("instruct_model")
instruction_activity=dbutils.widgets.get("instruction_activity")
df = spark.read.table(f"{catalog}.{schema}.topic_info_local")

In [0]:
%run ./config/instruction_configuration

In [0]:
column_name=instructions[instruction_activity]["out_column_name"]
ai_query_logic=instructions[instruction_activity]["ai_query_logic"]
model_result_table=instructions[instruction_activity]["output_table"]
source_table=instructions[instruction_activity]["input_table"]
key_cols=instructions[instruction_activity]["key_cols"] if "key_cols" in instructions[instruction_activity] else default_key_cols
result_column_schema=instructions[instruction_activity]["result_column_schema"]
target_table_name=f"{catalog}.{schema}.{model_result_table}"
source_table_name=f"{catalog}.{schema}.{source_table}"
additional_cols=instructions[instruction_activity]["additional_cols"] if "additional_cols" in instructions[instruction_activity] else []
checkpoint_path = f"/Volumes/{catalog}/{schema}/checkpoints/{model_result_table}.txt"
print(f"table_name:{target_table_name}, instruct_model:{instruct_model}, instruction_activity: {instruction_activity}" )


In [0]:
from pyspark.sql.functions import lit

key_col_list = [col.strip() for col in key_cols.split(",")]
additional_col_list = [col.strip() for col in additional_cols.split(",")] if additional_cols else []
selected_columns = key_col_list + additional_col_list
select_expr = ", ".join(selected_columns)

changes_df = None
last_version = read_checkpoint(checkpoint_path)
current_version = get_current_version(source_table_name)
if last_version is None:
    # Bootstrap mode:
    # source may have existed before CDF was enabled, so initialize from full table
    changes_df = (
        spark.table(source_table_name)
        .select(*selected_columns)
        .dropDuplicates(key_col_list)
    )

elif last_version >= current_version:
    dbutils.notebook.exit(
        f"No new rows to process. Checkpoint={last_version}, latest={current_version}"
    )

else:
    try:
        changes_df = (
            spark.sql(f"""
                SELECT {select_expr}
                FROM table_changes('{source_table_name}', {last_version + 1}, {current_version})
                WHERE _change_type = 'insert'
            """)
            .dropDuplicates(key_col_list)
        )
    except Exception as e:
        msg = str(e).lower()

        # Fallback for cases where CDF was not enabled for the requested history
        if "change data feed" in msg or "start version" in msg or "cdf" in msg:
            changes_df = (
                spark.table(source_table_name)
                .select(*selected_columns)
                .dropDuplicates(key_col_list)
            )
        else:
            raise

# Create target table if missing
if not spark.catalog.tableExists(target_table_name):
    empty_df = (
        spark.table(source_table_name)
        .select(*key_col_list)
        .withColumn(f"{column_name}", lit(None).cast("string"))
        .limit(0)
    )

    empty_df.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .saveAsTable(target_table_name)

if changes_df.limit(1).count() == 0:
    write_checkpoint(checkpoint_path, current_version)
    dbutils.notebook.exit(
        f"No new rows to process from source table. Checkpoint updated to {current_version}"
    )

changes_df.createOrReplaceTempView("changes_to_score")

print(f"Prepared changes_to_score with source versions through {current_version}")
display(changes_df.limit(10))


In [0]:
def wrap_ai_json(ai_query_logic: str, schema: str) -> str:
    return f"""
    from_json(
    regexp_replace(
        regexp_replace(
        {ai_query_logic},
        '^```[a-zA-Z]*\\\\s*',
        ''
        ),
        '\\\\s*```$',
        ''
    ),
    '{schema}'
    )
    """
#wrapped_ai_query_logic = wrap_ai_json(
#    ai_query_logic,
#    "title STRING, description STRING"
#)


wrapped_ai_query_logic = f"""
to_json(
from_json(
    regexp_replace(
    regexp_replace(
        {ai_query_logic},
        '^```[a-zA-Z]*\\\\s*',
        ''
    ),
    '\\\\s*```$',
    ''
    ),
    '{result_column_schema}'
)
)
"""

In [0]:
wrapped_ai_query_logic

In [0]:
score_select_expr = ",\n".join(key_col_list)
scored_chnages_sql = f"""
SELECT
{score_select_expr},
{wrapped_ai_query_logic} AS {column_name}
FROM changes_to_score
"""
print(scored_chnages_sql)
scored_df = spark.sql(scored_chnages_sql)
scored_df.createOrReplaceTempView("scored_changes")

In [0]:
%sql
select * from scored_changes

In [0]:
merge_on_expr = "\n    AND ".join([f"target.{col} <=> source.{col}" for col in key_col_list])
insert_columns_expr = ",\n    ".join(key_col_list + [column_name])
insert_values_expr = ",\n    ".join([f"source.{col}" for col in key_col_list + [column_name]])
merge_sql = f"""
MERGE INTO {target_table_name} AS target
USING scored_changes AS source
ON {merge_on_expr}
WHEN MATCHED THEN
UPDATE SET
    target.{column_name} = source.{column_name}
WHEN NOT MATCHED THEN
INSERT (
    {insert_columns_expr}
)
VALUES (
    {insert_values_expr}
)
"""
print(merge_sql)
spark.sql(merge_sql)
write_checkpoint(checkpoint_path, current_version)

In [0]:
spark.table(target_table_name).display()